# 22.2 — Compare Countries Using Distance Metrics / 使用距离度量比较国家

**Chapter 22 — File 2 of 2 / 第22章 — 第2个文件（共2个）**

## Summary / 总结

Compute Euclidean and cosine distances between countries based on economic indicators. Demonstrates how vector distances can be used to find similar countries and analyze economic relationships.

基于经济指标计算国家之间的欧几里得距离和余弦距离。演示如何使用向量距离来查找相似国家和分析经济关系。

## Distance Metrics / 距离度量

1. **Euclidean Distance**: $d(x,y) = \sqrt{\sum (x_i - y_i)^2}$ (raw difference)
2. **Cosine Similarity**: $\cos(\theta) = \frac{x \cdot y}{||x|| ||y||}$ (relative direction)

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import necessary libraries
# 导入必要的库
from pandas_datareader import wb
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances

## Step 2 — Download and Prepare Data / 下载和准备数据

In [ ]:
# Download World Bank data
# 下载世界银行数据
names = [
    "NE.EXP.GNFS.CD",  # Exports
    "NE.IMP.GNFS.CD",  # Imports
    "NV.AGR.TOTL.CD",  # Agriculture
    "NY.GDP.MKTP.CD",  # GDP
    "NE.RSB.GNFS.CD",  # External balance
]

try:
    df = wb.download(
        country='all',
        indicator=names,
        start=2010,
        end=2010
    ).reset_index()
    
    # Filter non-aggregate countries
    countries = wb.get_countries()
    non_aggregates = countries[countries['region'] != 'Aggregates'].name
    df = df[df['country'].isin(non_aggregates)].dropna()
    
    print(f"Data loaded: {df.shape}")
except Exception as e:
    print(f"Note: Could not download data: {e}")
    print(f"This notebook requires internet connection to World Bank API")

## Step 3 — Prepare Feature Matrix / 准备特征矩阵

In [ ]:
# Extract feature matrix and country names
# 提取特征矩阵和国家名称
X = df.iloc[:, 2:].values  # Exclude year and country columns
country_names = df['country'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Number of countries: {len(country_names)}")
print(f"Number of features: {X.shape[1]}")

# Show first few countries
print(f"\nFirst 5 countries: {country_names[:5]}")

## Step 4 — Scale Features / 缩放特征

In [ ]:
# Standardize features (important for distance metrics)
# 标准化特征（对距离度量很重要）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Mean of scaled features: {X_scaled.mean(axis=0)}")
print(f"Std of scaled features: {X_scaled.std(axis=0)}")

## Step 5 — Compute Euclidean Distances / 计算欧几里得距离

In [ ]:
# Compute pairwise Euclidean distances
# 计算两两欧几里得距离
euclidean_dist = euclidean_distances(X_scaled)

print(f"Euclidean distance matrix shape: {euclidean_dist.shape}")
print(f"\nDistance statistics:")
print(f"  Min distance: {euclidean_dist[euclidean_dist > 0].min():.4f}")
print(f"  Max distance: {euclidean_dist.max():.4f}")
print(f"  Mean distance: {euclidean_dist[euclidean_dist > 0].mean():.4f}")

## Step 6 — Compute Cosine Distances / 计算余弦距离

In [ ]:
# Compute cosine distances (1 - cosine similarity)
# 计算余弦距离
cosine_dist = cosine_distances(X_scaled)

print(f"Cosine distance matrix shape: {cosine_dist.shape}")
print(f"\nDistance statistics:")
print(f"  Min distance: {cosine_dist[cosine_dist > 0].min():.4f}")
print(f"  Max distance: {cosine_dist.max():.4f}")
print(f"  Mean distance: {cosine_dist[cosine_dist > 0].mean():.4f}")

## Step 7 — Find Similar Countries to Australia / 查找与澳大利亚相似的国家

In [ ]:
# Find index of a specific country (Australia)
# 查找特定国家（澳大利亚）的索引
try:
    australia_idx = np.where(country_names == 'Australia')[0][0]
    
    # Get distances from Australia to all other countries
    # 获取从澳大利亚到所有其他国家的距离
    distances_to_australia_euclidean = euclidean_dist[australia_idx]
    distances_to_australia_cosine = cosine_dist[australia_idx]
    
    # Find 10 closest countries (excluding Australia itself)
    # 查找10个最近的国家（排除澳大利亚本身）
    n_closest = 10
    
    # Euclidean distances
    euclidean_closest = np.argsort(distances_to_australia_euclidean)[1:n_closest+1]
    print("Countries most similar to Australia (by Euclidean distance):")
    for i, idx in enumerate(euclidean_closest, 1):
        print(f"  {i}. {country_names[idx]}: {distances_to_australia_euclidean[idx]:.4f}")
    
    # Cosine distances
    cosine_closest = np.argsort(distances_to_australia_cosine)[1:n_closest+1]
    print(f"\nCountries most similar to Australia (by Cosine distance):")
    for i, idx in enumerate(cosine_closest, 1):
        print(f"  {i}. {country_names[idx]}: {distances_to_australia_cosine[idx]:.4f}")
        
except Exception as e:
    print(f"Could not find Australia: {e}")

## Step 8 — Compare Distance Metrics / 比较距离度量

In [ ]:
# Discuss differences between Euclidean and Cosine distances
# 讨论欧几里得距离和余弦距离的差异
print(f"\nDistance Metric Comparison:")
print(f"\nEuclidean Distance:")
print(f"  - Measures absolute differences between vectors")
print(f"  - Sensitive to scale of features")
print(f"  - Geometric distance in feature space")
print(f"\nCosine Distance:")
print(f"  - Measures angle between vectors")
print(f"  - Invariant to scale (measures direction only)")
print(f"  - Better for comparing vector direction/pattern")
print(f"\nFor economic data:")
print(f"  - Euclidean: Countries with similar absolute sizes")
print(f"  - Cosine: Countries with similar economic structure/proportions")

## Learning Notes / 学习笔记

- **Math Essence**: Euclidean distance measures absolute differences (affected by scale), while cosine distance measures angular similarity (scale-invariant). Choosing the right metric depends on whether you care about absolute magnitude or relative direction.
  
  **数学本质**：欧几里得距离衡量绝对差异（受尺度影响），而余弦距离衡量角度相似性（尺度不变）。选择正确的度量取决于是否关心绝对大小或相对方向。

- **ML Application**: (1) Distance metrics are fundamental to clustering, nearest-neighbors, and similarity search, (2) Feature scaling is crucial for distance-based algorithms - StandardScaler normalizes each feature to mean=0, std=1, ensuring equal importance, (3) Different metrics can produce different results (see Australia example), choose based on domain knowledge of what "similarity" means for your data.
  
  **ML应用**：(1) 距离度量对聚类、最近邻和相似性搜索至关重要，(2) 特征缩放对基于距离的算法至关重要，(3) 不同度量可产生不同结果，根据领域知识选择。

➡️ **Next / 下一步**: `../chapter_23/01_show_tar.ipynb` — Working with compressed tar archives

## Complete Code / 完整代码一览

In [ ]:
# --- Import Section / 导入部分 ---
from pandas_datareader import wb
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances

# --- Download Data / 下载数据 ---
names = [
    "NE.EXP.GNFS.CD",
    "NE.IMP.GNFS.CD",
    "NV.AGR.TOTL.CD",
    "NY.GDP.MKTP.CD",
    "NE.RSB.GNFS.CD",
]
df = wb.download(country='all', indicator=names, start=2010, end=2010).reset_index()
countries = wb.get_countries()
non_aggregates = countries[countries['region'] != 'Aggregates'].name
df = df[df['country'].isin(non_aggregates)].dropna()

# --- Prepare Features / 准备特征 ---
X = df.iloc[:, 2:].values
country_names = df['country'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Compute Distances / 计算距离 ---
euclidean_dist = euclidean_distances(X_scaled)
cosine_dist = cosine_distances(X_scaled)

# --- Find Similar Countries / 查找相似国家 ---
australia_idx = np.where(country_names == 'Australia')[0][0]
closest = np.argsort(euclidean_dist[australia_idx])[1:11]
for idx in closest:
    print(f"{country_names[idx]}: {euclidean_dist[australia_idx, idx]:.4f}")